# Follow-ups on the drug reviews run

Two follow-up analyses on top of `quickstart_drug_reviews.ipynb` (same data sample, split, embeddings and SAE settings):

- **Part A: re-interpreting the most predictive neuron.** In the main run, the neuron most correlated with the label (r = 0.66) was labeled "mentions a specific dosage in milligrams". That label had poor fidelity (F1 = 0.29) and the hypothesis did not predict held-out ratings (p = 0.94). Here we generate 3 candidate labels for that neuron, once with the task instructions from the main run (which ask for specific content rather than general satisfaction) and once with neutral instructions, and check whether a better label recovers the neuron's predictive signal.
- **Part B: embedding similarity vs. LLM annotation.** Annotating 1,200 held-out reviews × 20 hypotheses with the LLM takes ~25 minutes. Here we compare it against a much cheaper alternative: cosine similarity between each review's embedding and each hypothesis's embedding. How well do the two agree, and do they lead to the same conclusions about which hypotheses are predictive?

Needs the `MODAL_VLLM_KEY` Colab secret. A T4 GPU runtime is faster, but a CPU runtime also works: the LLM runs on Modal, and only the embeddings and the small SAE run in Colab (timings in Part B then reflect CPU embedding).

In [ ]:
!pip install -q git+https://github.com/psnandini/HypotheSAEs.git
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: running on CPU'

In [ ]:
import os
from google.colab import userdata, files

os.environ["OPENAI_BASE_URL"] = "https://psnandini--hypothesaes-vllm-serve.us-east.modal.run/v1"
os.environ["OPENAI_KEY_SAE"] = userdata.get("MODAL_VLLM_KEY")
os.environ["EMB_CACHE_DIR"] = "/content/emb_cache"

import html, time
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer

from hypothesaes.quickstart import train_sae
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.llm_api import get_completion
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, LLMConfig, SamplingConfig
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses, compute_ols_metrics

MODEL = "Qwen/Qwen3-8B"
ANNOTATION_KWARGS = {"prompt_suffix": " /no_think"}   # thinking off for yes/no annotation
OUT = "/content/results"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.max_colwidth", None)

t = time.time()
print(get_completion(prompt="Reply with the single word: ready /no_think", model=MODEL), f"({time.time() - t:.0f} s)")

## Same data, split, embeddings and SAE as the main run

This cell repeats the main notebook's sampling (fixed random seeds), so the train / held-out reviews are identical.

In [ ]:
!wget -q -nc https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
!unzip -q -o drugsCom_raw.zip -d drugs

df = pd.concat([pd.read_csv(f"drugs/{f}", sep="\t") for f in ["drugsComTrain_raw.tsv", "drugsComTest_raw.tsv"]])
df["review"] = df.review.map(html.unescape).str.strip().str.strip('"').str.strip()
df = df.drop_duplicates(subset="review")
df = df[(df.rating >= 9) | (df.rating <= 4)].copy()
df["label"] = (df.rating >= 9).astype(int)
df = df.groupby("label").sample(n=3000, random_state=0).sample(frac=1, random_state=0).reset_index(drop=True)
n = len(df)
df["split"] = np.select([df.index < 0.7 * n, df.index < 0.8 * n], ["train", "val"], "holdout")
train_df, val_df, holdout_df = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "holdout"])
texts, labels = train_df.review.tolist(), train_df.label.values
holdout_texts, holdout_labels = holdout_df.review.tolist(), holdout_df.label.values
print(len(texts), len(val_df), len(holdout_texts))

EMBEDDER = "nomic-ai/modernbert-embed-base"
text2embedding = get_local_embeddings(texts + val_df.review.tolist(), model=EMBEDDER, batch_size=64, cache_name="drugs_modernbert")
train_embeddings = np.stack([text2embedding[t] for t in texts])
val_embeddings = np.stack([text2embedding[t] for t in val_df.review])

# Hypotheses from the main run
results = pd.read_csv("https://raw.githubusercontent.com/psnandini/HypotheSAEs/main/results/drugs_hypotheses.csv")
hypotheses = results.interpretation.tolist()

SAE: a new Colab notebook runs on a fresh machine, so the main run's SAE checkpoint is not available here and the SAE is retrained with the same settings (now with a fixed seed, so this notebook is reproducible). Neuron numbers therefore differ from the main run. Part A uses the neuron most correlated with the label in this SAE; its correlation is printed for comparison with the main run's top neuron (r = 0.66). The main run's label for that neuron ("mentions a specific dosage in milligrams") is a plain-text hypothesis, so it is tested on held-out data regardless.

In [ ]:
torch.manual_seed(0); np.random.seed(0)
sae = train_sae(embeddings=train_embeddings, M=256, K=8, matryoshka_prefix_lengths=[32, 256],
                checkpoint_dir="/content/checkpoints/drugs_followup", val_embeddings=val_embeddings)

activations = sae.get_activations(torch.tensor(train_embeddings, dtype=torch.float))
corrs = np.array([pearsonr(activations[:, i], labels)[0] if activations[:, i].std() > 0 else 0 for i in range(activations.shape[1])])
NEURON = int(np.nanargmax(np.abs(corrs)))
print(f"Neuron {NEURON}: correlation with label {corrs[NEURON]:+.3f}, active in {(activations[:, NEURON] > 0).mean():.1%} of training reviews")

## Part A: what does the most predictive neuron actually encode?

First, look at the reviews it fires on most strongly, and compare active vs. inactive reviews on simple properties.

In [ ]:
act = activations[:, NEURON]
active = act > 0
n_words = np.array([len(t.split()) for t in texts])
print(f"positive-review rate: active {labels[active].mean():.2f} vs inactive {labels[~active].mean():.2f}")
print(f"median words:         active {np.median(n_words[active]):.0f} vs inactive {np.median(n_words[~active]):.0f}")
print(f"mentions 'mg':        active {np.mean(['mg' in t.lower() for t in np.array(texts)[active]]):.2f} vs inactive {np.mean(['mg' in t.lower() for t in np.array(texts)[~active]]):.2f}")
print()
for rank, i in enumerate(np.argsort(act)[::-1][:8], 1):
    print(f"{rank}. [{'positive' if labels[i] else 'negative'}] {texts[i][:300]}")
    print()

Generate 3 candidate labels under each instruction set, then score every candidate's fidelity on 100 training reviews (50 where the neuron fires most, 50 where it is silent).

In [ ]:
ORIGINAL_INSTRUCTIONS = """All of the texts are patient reviews of a medication, posted on Drugs.com.
Features should describe a specific property of the review's content, such as a particular symptom, side effect, or experience,
rather than general satisfaction with the drug. For example:
- "mentions a specific dosage in milligrams"
- "mentions taking the medication for several years"
"""
NEUTRAL_INSTRUCTIONS = """All of the texts are patient reviews of a medication, posted on Drugs.com.
Features should describe a specific property of the review's content or writing."""

interpreter = NeuronInterpreter(interpreter_model=MODEL, annotator_model=MODEL,
                                n_workers_interpretation=8, n_workers_annotation=32)
candidates = []
for name, instructions in [("original", ORIGINAL_INSTRUCTIONS), ("neutral", NEUTRAL_INSTRUCTIONS)]:
    config = InterpretConfig(sampling=SamplingConfig(n_examples=20, max_words_per_example=256),
                             llm=LLMConfig(temperature=0.7), n_candidates=3,
                             task_specific_instructions=instructions)
    interps = interpreter.interpret_neurons(texts=texts, activations=activations, neuron_indices=[NEURON], config=config)
    candidates += [(name, c) for c in interps[NEURON] if c]

scores = interpreter.score_interpretations(texts=texts, activations=activations,
                                           interpretations={NEURON: [c for _, c in candidates]},
                                           config=ScoringConfig(n_examples=100), **ANNOTATION_KWARGS)
cand_df = pd.DataFrame([{"instructions": name, "interpretation": c, **scores[NEURON][c]} for name, c in candidates])
cand_df = cand_df.sort_values("f1", ascending=False).reset_index(drop=True)
display(cand_df)

Now test on held-out reviews: does the best new label predict ratings better than the original "milligrams" label? (Each label is tested on its own, so the numbers are not affected by the other 19 hypotheses.)

In [ ]:
old_label = results.loc[results.neuron_idx == 5, "interpretation"].item()   # main run's top neuron
test_labels = [old_label, cand_df.interpretation[0]]
ann = annotate_texts_with_concepts(texts=holdout_texts, concepts=test_labels, model=MODEL, cache_name="drugs_followup",
                                   max_words_per_example=256, n_workers=32, **ANNOTATION_KWARGS)
rows = []
for h in test_labels:
    m, table = score_hypotheses({h: ann[h]}, holdout_labels, classification=True)
    rows.append({"label": h, "source": "main run label" if h == old_label else "best new candidate",
                 "held-out AUROC": m["auroc"], **table.iloc[0].drop("hypothesis").to_dict()})
partA = pd.DataFrame(rows)
display(partA)
cand_df.to_csv(f"{OUT}/followup_neuron_candidates.csv", index=False)
partA.to_csv(f"{OUT}/followup_neuron_holdout.csv", index=False)
files.download(f"{OUT}/followup_neuron_candidates.csv")
files.download(f"{OUT}/followup_neuron_holdout.csv")

## Part B: embedding similarity vs. LLM annotation

**LLM annotation** (as in the main run): Qwen3-8B answers yes/no for each of the 1,200 held-out reviews × 20 hypotheses. This is the slow step (~25 min); the answers are saved right away.

In [ ]:
t = time.time()
llm_ann = annotate_texts_with_concepts(texts=holdout_texts, concepts=hypotheses, model=MODEL, cache_name="drugs_followup",
                                       max_words_per_example=256, n_workers=32, **ANNOTATION_KWARGS)
llm_seconds = time.time() - t
llm_df = pd.DataFrame(llm_ann)
llm_df.to_csv(f"{OUT}/followup_llm_annotations.csv", index=False)
files.download(f"{OUT}/followup_llm_annotations.csv")
print(f"LLM annotation: {llm_seconds / 60:.1f} min")

**Embedding similarity:** embed each review and each hypothesis with the same embedding model, and score each (review, hypothesis) pair by cosine similarity. `modernbert-embed-base` was trained for retrieval with `search_query:` / `search_document:` prefixes, so the hypothesis is treated as the query.

In [ ]:
t = time.time()
encoder = SentenceTransformer(EMBEDDER, device="cuda" if torch.cuda.is_available() else "cpu")
doc_emb = encoder.encode(["search_document: " + x for x in holdout_texts], batch_size=64, normalize_embeddings=True)
hyp_emb = encoder.encode(["search_query: " + h for h in hypotheses], normalize_embeddings=True)
cos = doc_emb @ hyp_emb.T            # (1200 reviews, 20 hypotheses)
emb_seconds = time.time() - t
print(f"Embedding similarity: {emb_seconds:.0f} s  vs  LLM annotation: {llm_seconds:.0f} s")

**1. Do they agree?** For each hypothesis: how well does the cosine score rank the reviews the LLM said "yes" to above the ones it said "no" to (AUROC; 0.5 = no agreement, 1 = perfect), and the correlation between the two.

In [ ]:
agree = []
for j, h in enumerate(hypotheses):
    y = llm_df[h].values
    agree.append({"hypothesis": h, "LLM prevalence": y.mean(),
                  "agreement AUROC": roc_auc_score(y, cos[:, j]) if 0 < y.mean() < 1 else np.nan,
                  "correlation": pearsonr(y, cos[:, j])[0] if 0 < y.mean() < 1 else np.nan})
agree_df = pd.DataFrame(agree).sort_values("agreement AUROC", ascending=False)
display(agree_df.round(3))
print(f"median agreement AUROC: {agree_df['agreement AUROC'].median():.3f}")

**2. Same conclusions?** Run the same held-out evaluation (logistic regression on all 20 hypotheses, Bonferroni p < 0.005) three ways:
- LLM yes/no annotations (as in the main run)
- cosine scores, binarized: for each hypothesis, the top-scoring reviews are marked "yes", with the same number of "yes" reviews as the LLM gave (so the two differ only in *which* reviews they mark)
- raw cosine scores (no binarization, no LLM information at all)

In [ ]:
binarized = {}
for j, h in enumerate(hypotheses):
    k = int(llm_df[h].sum())
    b = np.zeros(len(holdout_texts), dtype=int)
    b[np.argsort(cos[:, j])[::-1][:k]] = 1
    binarized[h] = b
zscored = {h: (cos[:, j] - cos[:, j].mean()) / cos[:, j].std() for j, h in enumerate(hypotheses)}

m_llm, t_llm = score_hypotheses(llm_ann, holdout_labels, classification=True)
m_bin, t_bin = score_hypotheses(binarized, holdout_labels, classification=True)
m_raw, coefs_raw = compute_ols_metrics(hypothesis_annotations=zscored, y_true=holdout_labels, classification=True)
threshold = 0.1 / len(hypotheses)

compare = pd.DataFrame({"hypothesis": hypotheses})
for name, coefs in [("LLM", dict(zip(t_llm.hypothesis, zip(t_llm.regression_coef, t_llm.regression_pval)))),
                    ("embedding (binarized)", dict(zip(t_bin.hypothesis, zip(t_bin.regression_coef, t_bin.regression_pval)))),
                    ("embedding (raw)", coefs_raw)]:
    compare[f"{name} coef"] = [coefs[h][0] for h in hypotheses]
    compare[f"{name} significant"] = [coefs[h][1] < threshold for h in hypotheses]
display(compare.round(3))

summary = pd.DataFrame([
    {"annotation method": "LLM (Qwen3-8B)", "held-out AUROC": m_llm["auroc"], "significant": m_llm["Significant"][0], "time (s)": llm_seconds},
    {"annotation method": "embedding, binarized", "held-out AUROC": m_bin["auroc"], "significant": m_bin["Significant"][0], "time (s)": emb_seconds},
    {"annotation method": "embedding, raw cosine", "held-out AUROC": m_raw["auroc"], "significant": sum(p < threshold for _, p in coefs_raw.values()), "time (s)": emb_seconds},
])
display(summary.round(3))
sig = {c: set(compare.hypothesis[compare[f"{c} significant"]]) for c in ["LLM", "embedding (binarized)", "embedding (raw)"]}
print("significant for both LLM and binarized embedding:", len(sig["LLM"] & sig["embedding (binarized)"]), "of", len(sig["LLM"]), "LLM-significant")
print("coefficient signs agree (LLM vs binarized):", int((np.sign(compare["LLM coef"]) == np.sign(compare["embedding (binarized) coef"])).sum()), "/ 20")

In [ ]:
agree_df.to_csv(f"{OUT}/followup_agreement.csv", index=False)
compare.to_csv(f"{OUT}/followup_llm_vs_embedding.csv", index=False)
summary.to_csv(f"{OUT}/followup_summary.csv", index=False)
for f in ["followup_agreement.csv", "followup_llm_vs_embedding.csv", "followup_summary.csv"]:
    files.download(f"{OUT}/{f}")